# A/B Testing with Chi-Square Test of Independence

## Notebook 3: Chi-Square Tests for Categorical Outcomes

**Purpose**: Use chi-square tests to determine if there's a statistically significant relationship between treatment group and categorical outcomes (visit, conversion).

### What is the Chi-Square Test?

The chi-square test examines whether **observed frequencies** in data match **expected frequencies** under the null hypothesis of independence.

**Core Question**: Is there an association between treatment group and outcome category, or are they independent?

- **H₀ (Null)**: Treatment and outcome are independent (no association)
- **H₁ (Alternative)**: Treatment and outcome are related (association exists)

### When to Use Chi-Square?

- Both variables are categorical
- Sample sizes are reasonably large (expected frequencies > 5 in most cells)
- Observations are independent
- It's a test of association/independence, not causation

This notebook covers chi-square tests for our binary outcomes: visit (yes/no) and conversion (yes/no).

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from scipy.stats import chi2
import warnings
warnings.filterwarnings('ignore')

# Try to import plotly
try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    print("Plotly not available - static plots will be used")

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Create output directory
os.makedirs('../data/outputs/nb03', exist_ok=True)


In [2]:
# Load cleaned dataset
df = pd.read_csv('../data/outputs/nb01/nb01_hillstrom_clean.csv')

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nTreatment groups: {df['segment'].unique()}")
print(f"\nOutcome variables: visit, conversion")

Dataset loaded: 64,000 rows × 24 columns

Treatment groups: ['Womens E-Mail' 'No E-Mail' 'Mens E-Mail']

Outcome variables: visit, conversion


## Chi-Square Test Basics

### Contingency Tables

A **contingency table** (or cross-tabulation) shows the joint distribution of two categorical variables.

**Example**: Visit × Treatment Group

|  | Mens Email | Womens Email | No Email | Total |
|---|---|---|---|---|
| No Visit (0) | 7,500 | 7,600 | 7,900 | 23,000 |
| Visit (1) | 8,500 | 8,400 | 8,100 | 25,000 |
| **Total** | **16,000** | **16,000** | **16,000** | **48,000** |

### Chi-Square Statistic

$$\chi^2 = \sum \frac{(O - E)^2}{E}$$

Where:
- O = observed frequency (actual data)
- E = expected frequency (what we'd expect if H₀ is true)

**Interpretation**:
- Large χ²: Observed differs greatly from expected → reject H₀
- Small χ²: Observed matches expected → fail to reject H₀

### Degrees of Freedom

$$df = (r - 1)(c - 1)$$

Where r = number of rows, c = number of columns

For a 2×3 table (2 outcomes × 3 groups): df = (2-1)(3-1) = 2

## Step 1: Build Contingency Tables

Let's create contingency tables for our two binary outcomes.

In [3]:
# Contingency table for VISIT
print("=" * 70)
print("CONTINGENCY TABLE: VISIT by TREATMENT GROUP")
print("=" * 70)

visit_table = pd.crosstab(df['segment'], df['visit'], margins=True)
print(visit_table)

# Add proportions
print("\nProportion within each group:")
visit_props = pd.crosstab(df['segment'], df['visit'], normalize='index')
print(visit_props.round(4))

CONTINGENCY TABLE: VISIT by TREATMENT GROUP
visit              0     1    All
segment                          
Mens E-Mail    17413  3894  21307
No E-Mail      19044  2262  21306
Womens E-Mail  18149  3238  21387
All            54606  9394  64000

Proportion within each group:
visit               0       1
segment                      
Mens E-Mail    0.8172  0.1828
No E-Mail      0.8938  0.1062
Womens E-Mail  0.8486  0.1514


In [4]:
# Contingency table for CONVERSION
print("\n" + "=" * 70)
print("CONTINGENCY TABLE: CONVERSION by TREATMENT GROUP")
print("=" * 70)

conv_table = pd.crosstab(df['segment'], df['conversion'], margins=True)
print(conv_table)

# Add proportions
print("\nProportion within each group:")
conv_props = pd.crosstab(df['segment'], df['conversion'], normalize='index')
print(conv_props.round(4))


CONTINGENCY TABLE: CONVERSION by TREATMENT GROUP
conversion         0    1    All
segment                         
Mens E-Mail    21040  267  21307
No E-Mail      21184  122  21306
Womens E-Mail  21198  189  21387
All            63422  578  64000

Proportion within each group:
conversion          0       1
segment                      
Mens E-Mail    0.9875  0.0125
No E-Mail      0.9943  0.0057
Womens E-Mail  0.9912  0.0088


## Step 2: Perform Chi-Square Test of Independence

For each outcome (visit and conversion), we'll test whether treatment group and outcome are independent using the chi-square test.

In [5]:
def perform_chi_square_test(contingency_table, outcome_name):
    """Perform chi-square test on a contingency table."""
    # Remove margins if present
    if 'All' in contingency_table.columns:
        ct = contingency_table.drop('All', axis=1)
    else:
        ct = contingency_table
    if 'All' in ct.index:
        ct = ct.drop('All', axis=0)
    else:
        ct = ct
    
    # Perform chi-square test
    chi2_stat, p_val, dof, expected_freq = chi2_contingency(ct)
    
    return {
        'outcome': outcome_name,
        'chi2': chi2_stat,
        'p_value': p_val,
        'dof': dof,
        'expected_freq': expected_freq,
        'contingency_table': ct
    }

# Test for VISIT
visit_ct = pd.crosstab(df['segment'], df['visit'])
visit_test = perform_chi_square_test(visit_ct, 'Visit')

print("=" * 70)
print("CHI-SQUARE TEST: TREATMENT × VISIT")
print("=" * 70)
print(f"\nNull Hypothesis (H₀): Treatment group and visit are independent")
print(f"Alternative Hypothesis (H₁): Treatment group and visit are related")
print(f"\nContingency Table:")
print(visit_ct)
print(f"\nChi-square statistic: {visit_test['chi2']:.4f}")
print(f"P-value: {visit_test['p_value']:.10f}")
print(f"Degrees of freedom: {visit_test['dof']}")
print(f"\nResult: {'REJECT H₀' if visit_test['p_value'] < 0.05 else 'FAIL TO REJECT H₀'} (α = 0.05)")
if visit_test['p_value'] < 0.05:
    print(f"Conclusion: Significant association between treatment and visit (p < 0.05)")
else:
    print(f"Conclusion: No significant association (p ≥ 0.05)")

CHI-SQUARE TEST: TREATMENT × VISIT

Null Hypothesis (H₀): Treatment group and visit are independent
Alternative Hypothesis (H₁): Treatment group and visit are related

Contingency Table:
visit              0     1
segment                   
Mens E-Mail    17413  3894
No E-Mail      19044  2262
Womens E-Mail  18149  3238

Chi-square statistic: 504.4607
P-value: 0.0000000000
Degrees of freedom: 2

Result: REJECT H₀ (α = 0.05)
Conclusion: Significant association between treatment and visit (p < 0.05)


In [6]:
# Test for CONVERSION
conv_ct = pd.crosstab(df['segment'], df['conversion'])
conv_test = perform_chi_square_test(conv_ct, 'Conversion')

print("\n" + "=" * 70)
print("CHI-SQUARE TEST: TREATMENT × CONVERSION")
print("=" * 70)
print(f"\nNull Hypothesis (H₀): Treatment group and conversion are independent")
print(f"Alternative Hypothesis (H₁): Treatment group and conversion are related")
print(f"\nContingency Table:")
print(conv_ct)
print(f"\nChi-square statistic: {conv_test['chi2']:.4f}")
print(f"P-value: {conv_test['p_value']:.10f}")
print(f"Degrees of freedom: {conv_test['dof']}")
print(f"\nResult: {'REJECT H₀' if conv_test['p_value'] < 0.05 else 'FAIL TO REJECT H₀'} (α = 0.05)")
if conv_test['p_value'] < 0.05:
    print(f"Conclusion: Significant association between treatment and conversion (p < 0.05)")
else:
    print(f"Conclusion: No significant association (p ≥ 0.05)")


CHI-SQUARE TEST: TREATMENT × CONVERSION

Null Hypothesis (H₀): Treatment group and conversion are independent
Alternative Hypothesis (H₁): Treatment group and conversion are related

Contingency Table:
conversion         0    1
segment                  
Mens E-Mail    21040  267
No E-Mail      21184  122
Womens E-Mail  21198  189

Chi-square statistic: 55.2580
P-value: 0.0000000000
Degrees of freedom: 2

Result: REJECT H₀ (α = 0.05)
Conclusion: Significant association between treatment and conversion (p < 0.05)


## Step 3: Check Chi-Square Assumptions

### Assumption 1: Expected Frequency > 5

For chi-square to be valid, most cells should have expected frequency ≥ 5. Let's verify this.

In [7]:
print("=" * 70)
print("EXPECTED FREQUENCIES: VISIT")
print("=" * 70)
expected_visit = visit_test['expected_freq']
print(expected_visit)
print(f"\nMinimum expected frequency: {expected_visit.min():.2f}")
print(f"Cells with expected frequency < 5: {(expected_visit < 5).sum()}")
print(f"Valid for chi-square? {'✓ Yes' if (expected_visit < 5).sum() == 0 else '✗ No'}")

print("\n" + "=" * 70)
print("EXPECTED FREQUENCIES: CONVERSION")
print("=" * 70)
expected_conv = conv_test['expected_freq']
print(expected_conv)
print(f"\nMinimum expected frequency: {expected_conv.min():.2f}")
print(f"Cells with expected frequency < 5: {(expected_conv < 5).sum()}")
print(f"Valid for chi-square? {'✓ Yes' if (expected_conv < 5).sum() == 0 else '✗ No'}")

EXPECTED FREQUENCIES: VISIT
[[18179.53190625  3127.46809375]
 [18178.6786875   3127.3213125 ]
 [18247.78940625  3139.21059375]]

Minimum expected frequency: 3127.32
Cells with expected frequency < 5: 0
Valid for chi-square? ✓ Yes

EXPECTED FREQUENCIES: CONVERSION
[[21114.57115625   192.42884375]
 [21113.5801875    192.4198125 ]
 [21193.84865625   193.15134375]]

Minimum expected frequency: 192.42
Cells with expected frequency < 5: 0
Valid for chi-square? ✓ Yes


## Step 4: Pairwise Chi-Square Tests

The overall test shows whether treatment is associated with outcome. Now let's perform pairwise comparisons between specific groups using chi-square tests.

**Comparisons**:
1. Mens Email vs Control
2. Womens Email vs Control
3. Mens Email vs Womens Email

In [8]:
def chi_square_2group(data, seg1, seg2, outcome):
    """Perform chi-square test for two groups."""
    d1 = data[data['segment'] == seg1][outcome]
    d2 = data[data['segment'] == seg2][outcome]
    
    ct = pd.crosstab([d1.index], d1)
    ct2 = pd.crosstab([d2.index], d2)
    
    combined_ct = pd.concat([
        pd.Series([d1.value_counts().get(0, 0), d1.value_counts().get(1, 0)]),
        pd.Series([d2.value_counts().get(0, 0), d2.value_counts().get(1, 0)])
    ], axis=1).T
    
    chi2, p_val, dof, expected = chi2_contingency(combined_ct)
    return chi2, p_val

print("=" * 70)
print("PAIRWISE CHI-SQUARE TESTS: CONVERSION")
print("=" * 70)

segments = ["Mens E-Mail", "Womens E-Mail", "No E-Mail"]
pairwise_results = []

# Men vs Control
chi2_mc, p_mc = chi_square_2group(df, "Mens E-Mail", "No E-Mail", 'conversion')
pairwise_results.append({
    'Comparison': "Mens Email vs Control",
    'Chi-Square': chi2_mc,
    'P-value': p_mc,
    'Significant': 'Yes' if p_mc < 0.05 else 'No'
})
print(f"Mens Email vs Control: χ² = {chi2_mc:.4f}, p = {p_mc:.6f}")

# Women vs Control
chi2_wc, p_wc = chi_square_2group(df, "Womens E-Mail", "No E-Mail", 'conversion')
pairwise_results.append({
    'Comparison': "Womens Email vs Control",
    'Chi-Square': chi2_wc,
    'P-value': p_wc,
    'Significant': 'Yes' if p_wc < 0.05 else 'No'
})
print(f"Womens Email vs Control: χ² = {chi2_wc:.4f}, p = {p_wc:.6f}")

# Men vs Women
chi2_mw, p_mw = chi_square_2group(df, "Mens E-Mail", "Womens E-Mail", 'conversion')
pairwise_results.append({
    'Comparison': "Mens Email vs Womens Email",
    'Chi-Square': chi2_mw,
    'P-value': p_mw,
    'Significant': 'Yes' if p_mw < 0.05 else 'No'
})
print(f"Mens Email vs Womens Email: χ² = {chi2_mw:.4f}, p = {p_mw:.6f}")

PAIRWISE CHI-SQUARE TESTS: CONVERSION
Mens Email vs Control: χ² = 53.7902, p = 0.000000
Womens Email vs Control: χ² = 13.8581, p = 0.000197
Mens Email vs Womens Email: χ² = 13.4359, p = 0.000247


In [9]:
# Apply Bonferroni correction to pairwise tests
num_pairwise = 3
alpha_bonf = 0.05 / num_pairwise

print("\n" + "=" * 70)
print("BONFERRONI CORRECTION FOR PAIRWISE TESTS")
print("=" * 70)
print(f"Number of pairwise comparisons: {num_pairwise}")
print(f"Adjusted α: {alpha_bonf:.6f}\n")

pw_df = pd.DataFrame(pairwise_results)
pw_df['Significant (Bonf)'] = pw_df['P-value'] < alpha_bonf
print(pw_df.to_string(index=False))


BONFERRONI CORRECTION FOR PAIRWISE TESTS
Number of pairwise comparisons: 3
Adjusted α: 0.016667

                Comparison  Chi-Square      P-value Significant  Significant (Bonf)
     Mens Email vs Control   53.790187 2.230841e-13         Yes                True
   Womens Email vs Control   13.858111 1.971440e-04         Yes                True
Mens Email vs Womens Email   13.435910 2.468532e-04         Yes                True


## Step 5: Effect Size - Cramér's V

While chi-square tells us if an association exists, **Cramér's V** quantifies the strength of the association.

### Formula

$$V = \sqrt{\frac{\chi^2}{n \times \min(k-1, r-1)}}$$

Where:
- χ² = chi-square statistic
- n = sample size
- k = number of columns
- r = number of rows

### Interpretation

- V = 0: No association
- 0 < V < 0.1: Very weak association
- 0.1 ≤ V < 0.3: Weak association
- 0.3 ≤ V < 0.5: Moderate association
- V ≥ 0.5: Strong association

In [10]:
def cramers_v(chi2_stat, n, min_dim):
    """Calculate Cramér's V effect size."""
    return np.sqrt(chi2_stat / (n * min_dim))

print("=" * 70)
print("CRAMÉR'S V: EFFECT SIZE FOR CHI-SQUARE")
print("=" * 70)

# For Visit
n = len(df)
min_dim_visit = min(visit_ct.shape[0] - 1, visit_ct.shape[1] - 1)
v_visit = cramers_v(visit_test['chi2'], n, min_dim_visit)

print(f"\nVISIT Outcome:")
print(f"  Cramér's V: {v_visit:.4f}")
print(f"  Interpretation: {'Very weak' if v_visit < 0.1 else 'Weak' if v_visit < 0.3 else 'Moderate' if v_visit < 0.5 else 'Strong'} association")

# For Conversion
min_dim_conv = min(conv_ct.shape[0] - 1, conv_ct.shape[1] - 1)
v_conv = cramers_v(conv_test['chi2'], n, min_dim_conv)

print(f"\nCONVERSION Outcome:")
print(f"  Cramér's V: {v_conv:.4f}")
print(f"  Interpretation: {'Very weak' if v_conv < 0.1 else 'Weak' if v_conv < 0.3 else 'Moderate' if v_conv < 0.5 else 'Strong'} association")

CRAMÉR'S V: EFFECT SIZE FOR CHI-SQUARE

VISIT Outcome:
  Cramér's V: 0.0888
  Interpretation: Very weak association

CONVERSION Outcome:
  Cramér's V: 0.0294
  Interpretation: Very weak association


## Step 6: Residual Analysis

Chi-square is an omnibus test - it tells us if association exists, but not where the differences lie.

**Standardized Residuals** show which cells contribute most to the chi-square statistic:

$$\text{Residual} = \frac{O - E}{\sqrt{E}}$$

Large residuals (|residual| > 2) indicate cells that differ significantly from expected.

In [11]:
def calculate_standardized_residuals(contingency_table):
    """Calculate standardized residuals for contingency table."""
    chi2_stat, p_val, dof, expected = chi2_contingency(contingency_table)
    standardized_resid = (contingency_table.values - expected) / np.sqrt(expected)
    return pd.DataFrame(standardized_resid, 
                       index=contingency_table.index, 
                       columns=contingency_table.columns)

print("=" * 70)
print("STANDARDIZED RESIDUALS: VISIT")
print("=" * 70)
resid_visit = calculate_standardized_residuals(visit_ct)
print(resid_visit.round(3))
print("\nLarge residuals (|value| > 2) indicate cells contributing most to association")

print("\n" + "=" * 70)
print("STANDARDIZED RESIDUALS: CONVERSION")
print("=" * 70)
resid_conv = calculate_standardized_residuals(conv_ct)
print(resid_conv.round(3))
print("\nLarge residuals (|value| > 2) indicate cells contributing most to association")

STANDARDIZED RESIDUALS: VISIT
visit              0       1
segment                     
Mens E-Mail   -5.685  13.707
No E-Mail      6.418 -15.474
Womens E-Mail -0.731   1.763

Large residuals (|value| > 2) indicate cells contributing most to association

STANDARDIZED RESIDUALS: CONVERSION
conversion         0      1
segment                    
Mens E-Mail   -0.513  5.376
No E-Mail      0.485 -5.077
Womens E-Mail  0.029 -0.299

Large residuals (|value| > 2) indicate cells contributing most to association


In [12]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

In [13]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

In [14]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

## Step 7: Summary of Chi-Square Results

Let's compile all chi-square test results into a comprehensive summary.

In [15]:
# Comprehensive chi-square results summary
chi_results = []

# Overall tests
chi_results.append({
    'Test': 'Overall: Treatment × Visit',
    'Chi-Square': f"{visit_test['chi2']:.4f}",
    'DF': visit_test['dof'],
    'P-value': f"{visit_test['p_value']:.6f}",
    'Significant (α=0.05)': 'Yes' if visit_test['p_value'] < 0.05 else 'No',
    "Cramér's V": f"{v_visit:.4f}",
    'Effect': 'Very weak' if v_visit < 0.1 else 'Weak'
})

chi_results.append({
    'Test': 'Overall: Treatment × Conversion',
    'Chi-Square': f"{conv_test['chi2']:.4f}",
    'DF': conv_test['dof'],
    'P-value': f"{conv_test['p_value']:.6f}",
    'Significant (α=0.05)': 'Yes' if conv_test['p_value'] < 0.05 else 'No',
    "Cramér's V": f"{v_conv:.4f}",
    'Effect': 'Very weak' if v_conv < 0.1 else 'Weak'
})

chi_df = pd.DataFrame(chi_results)
print("=" * 70)
print("CHI-SQUARE TEST SUMMARY")
print("=" * 70)
print(chi_df.to_string(index=False))

# Save results
chi_df.to_csv('../data/outputs/nb03/nb03_chi_square_results.csv', index=False)
print("\n✓ Saved: ../data/outputs/nb03/nb03_chi_square_results.csv")

CHI-SQUARE TEST SUMMARY
                           Test Chi-Square  DF  P-value Significant (α=0.05) Cramér's V    Effect
     Overall: Treatment × Visit   504.4607   2 0.000000                  Yes     0.0888 Very weak
Overall: Treatment × Conversion    55.2580   2 0.000000                  Yes     0.0294 Very weak

✓ Saved: ../data/outputs/nb03/nb03_chi_square_results.csv


## Step 8: Chi-Square Assumptions Summary

### Assumption 1: Expected Frequencies
- ✓ All expected frequencies > 5 (verified above)
- Large sample size ensures stability of chi-square test

### Assumption 2: Independence of Observations
- ✓ Random assignment ensures independence between groups
- ✓ Each customer appears only once in dataset

### Assumption 3: Sample Size
- ✓ Sample size (n ≈ 64,000) is very large
- Large samples increase statistical power to detect true effects

### Conclusion on Assumptions
All chi-square assumptions are satisfied. Tests results are valid and reliable.

## How to Interpret Chi-Square Results

### If P-value < 0.05 (Reject H₀):
- **Finding**: Treatment group and outcome are statistically associated
- **Meaning**: Email campaign (treatment) is related to visit/conversion rates
- **Action**: Examine effect size (Cramér's V) to assess practical significance
  - Small effect (V < 0.1): Association exists but is weak
  - Large effect (V ≥ 0.3): Association is strong and practically important

### If P-value ≥ 0.05 (Fail to Reject H₀):
- **Finding**: No statistically significant association detected
- **Meaning**: Evidence insufficient to conclude treatment affects outcomes
- **Caution**: Absence of evidence ≠ evidence of absence
  - Could be underpowered (small effect size)
  - Could be no real effect

### Effect Size Interpretation (Cramér's V):
- **Very Weak** (V < 0.1): Minimal practical difference
- **Weak** (0.1 ≤ V < 0.3): Small but noticeable difference
- **Moderate** (0.3 ≤ V < 0.5): Substantial difference
- **Strong** (V ≥ 0.5): Large, important difference

## Key Takeaways

1. **Chi-Square Tests**:
   - Test for association between categorical variables
   - Compare observed vs. expected frequencies under independence
   - Produce chi-square statistic and p-value
   - Omnibus test - doesn't show which groups differ

2. **Contingency Tables**:
   - Show joint distribution of two categorical variables
   - Foundation for chi-square calculations
   - Proportions reveal pattern differences between groups

3. **Pairwise Tests**:
   - Follow-up comparisons after significant overall test
   - Use Bonferroni correction to control false positives
   - Identify which group pairs differ significantly

4. **Effect Size (Cramér's V)**:
   - Quantifies strength of association (0 to 1)
   - Statistical significance ≠ practical significance
   - Small effect: p < 0.05 but V < 0.1 (weak practical importance)

5. **Residual Analysis**:
   - Standardized residuals show which cells drive the association
   - Large residuals (|value| > 2) = cells differing from expectation
   - Helps identify which outcomes/groups differ most

6. **Assumptions**:
   - Expected frequencies > 5 in most cells
   - Independent observations (satisfied by random assignment)
   - Large sample sizes provide stable estimates

---

## Comprehensive A/B Testing Summary

Across three notebooks, we've covered:

1. **Notebook 1 - EDA**: Explore data, assess balance, understand outcomes
2. **Notebook 2 - Frequentist Tests**: Z-tests and t-tests for proportions and means
3. **Notebook 3 - Chi-Square**: Test categorical associations, effect sizes, residuals

Together, these provide a complete frequentist framework for A/B testing analysis.

---

## Blog-Ready Plotly Charts

The cells below regenerate the charts from this notebook as responsive Plotly
HTML files for embedding in the blog post. They are **self-contained**: each
one re-loads the clean dataset from nb01 and re-derives the statistics it
needs, so you can run this section in isolation.

Outputs are written to `data/outputs/nb##/` with the suffix `_interactive.html`.

**Required packages:** `plotly` (install with `pip install plotly` if missing).

In [16]:
# ============================================================
# Blog-Ready Plotly Charts — self-contained, embed-friendly
# ============================================================
# These cells produce responsive Plotly HTML files for the blog post.
# They re-load from the nb01 clean CSV and re-derive stats so the section
# runs standalone. Each figure uses:
#   - include_plotlyjs='cdn' (single shared CDN load on the blog page)
#   - config={'responsive': True} so it resizes to container width
#   - automargin=True on axes + generous margins so labels never clip
#   - rotated tick labels on long categories, headroom for outside labels
import os, numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook_connected"  # inline-render Plotly in cell output

OUT_DIR = os.path.abspath("../data/outputs/nb03")
os.makedirs(OUT_DIR, exist_ok=True)
CLEAN_CSV = os.path.abspath("../data/outputs/nb01/nb01_hillstrom_clean.csv")
df_blog = pd.read_csv(CLEAN_CSV)

# Shared palette aligned with nb01 Plotly charts
COLORS = {
    "Mens E-Mail": "#4C8BB8", "Womens E-Mail": "#5FA85F", "No E-Mail": "#E89B4C",
    "Match": "#2ECC71", "Mismatch": "#E74C3C", "Mixed": "#F39C12", "Control": "#95A5A6",
    "Treatment (Any Email)": "#4C8BB8",
}
PLOTLY_KW = dict(include_plotlyjs="cdn", full_html=True,
                 config={"responsive": True, "displaylogo": False})
BASE_LAYOUT = dict(template="plotly_white",
                   font=dict(family="Arial, sans-serif", size=13),
                   title_x=0.5,
                   margin=dict(l=70, r=40, t=90, b=90),
                   hoverlabel=dict(bgcolor="white", font_size=12))
print(f"Blog-ready Plotly charts will be written to: {OUT_DIR}")


Blog-ready Plotly charts will be written to: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/ab_testing/data/outputs/nb03


In [17]:
# Chart 1: Contingency table heatmap (segment × visit)
ct = pd.crosstab(df_blog["segment"], df_blog["visit"])
ct.columns = ["No Visit (0)", "Visit (1)"]
fig = go.Figure(go.Heatmap(
    z=ct.values, x=ct.columns, y=ct.index,
    text=[[f"{v:,}" for v in row] for row in ct.values], texttemplate="%{text}",
    colorscale="Blues", colorbar=dict(title="Count"),
    hovertemplate="<b>%{y}</b><br>%{x}: %{z:,}<extra></extra>"))
fig.update_layout(**BASE_LAYOUT, title="Contingency Table: Segment × Visit",
                  height=430,
                  xaxis=dict(automargin=True), yaxis=dict(automargin=True))
fig.write_html(os.path.join(OUT_DIR, "nb03_contingency_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb03_contingency_interactive.html")


  ✓ nb03_contingency_interactive.html


In [18]:
# Chart 2: Observed vs Expected side-by-side heatmaps
from scipy.stats import chi2_contingency as _chi2c
chi2, p_val, dof, expected = _chi2c(ct.values)
obs = ct.values; exp = expected

fig = make_subplots(rows=1, cols=2, subplot_titles=("Observed", "Expected"),
                    horizontal_spacing=0.15)
fig.add_trace(go.Heatmap(z=obs, x=ct.columns, y=ct.index,
                         text=[[f"{v:,}" for v in r] for r in obs], texttemplate="%{text}",
                         colorscale="Blues", showscale=False,
                         hovertemplate="Observed: %{z:,}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Heatmap(z=exp, x=ct.columns, y=ct.index,
                         text=[[f"{v:,.0f}" for v in r] for r in exp], texttemplate="%{text}",
                         colorscale="Oranges", showscale=False,
                         hovertemplate="Expected: %{z:,.1f}<extra></extra>"), row=1, col=2)
fig.update_xaxes(automargin=True); fig.update_yaxes(automargin=True)
fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=90, r=40, t=100, b=70)},
                  title=f"Observed vs Expected — χ²({dof}) = {chi2:.2f}, p = {p_val:.2e}",
                  height=450)
fig.write_html(os.path.join(OUT_DIR, "nb03_obs_vs_exp_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb03_obs_vs_exp_interactive.html")

# Chart 3: Standardized residuals heatmap — shows WHICH cells drive significance
resid = (obs - exp) / np.sqrt(exp)
fig = go.Figure(go.Heatmap(
    z=resid, x=ct.columns, y=ct.index,
    text=[[f"{v:+.2f}" for v in r] for r in resid], texttemplate="%{text}",
    colorscale="RdBu_r", zmid=0, colorbar=dict(title="Std.<br>Resid."),
    hovertemplate="<b>%{y}</b><br>%{x}<br>Std. residual: %{z:.2f}<extra></extra>"))
fig.update_layout(**BASE_LAYOUT,
                  title="Standardized Residuals — Cells with |r| > 2 Drive Significance",
                  height=450,
                  xaxis=dict(automargin=True), yaxis=dict(automargin=True))
fig.write_html(os.path.join(OUT_DIR, "nb03_std_residuals_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb03_std_residuals_interactive.html")


  ✓ nb03_obs_vs_exp_interactive.html


  ✓ nb03_std_residuals_interactive.html


### Results-Display Tables (embed-ready go.Table cards for every printed output)

Each card below mirrors one of the printed console blocks and saves as its own
HTML file under `../data/outputs/nb03/` for direct blog embedding.


In [19]:
# Results-display: Plotly go.Table cards for every printed chi-square output
from scipy.stats import chi2_contingency, chi2 as chi2_dist

SEGMENT_ORDER = ["Mens E-Mail", "No E-Mail", "Womens E-Mail"]

def table_card(title, header_vals, cell_cols, colwidths, row_colors=None,
               cell_font_size=12, height_extra=60):
    n_rows = len(cell_cols[0]) if cell_cols else 0
    stripe = ["#F8F9F9" if i%2==0 else "white" for i in range(n_rows)]
    fill = [row_colors[i] if row_colors else stripe for i in range(len(cell_cols))]
    fig = go.Figure(data=[go.Table(
        columnwidth=colwidths,
        header=dict(values=[f"<b>{h}</b>" for h in header_vals],
                    fill_color="#2C3E50",
                    font=dict(color="white", size=13),
                    align="center", height=36),
        cells=dict(values=cell_cols, fill_color=fill,
                   align="center",
                   font=dict(size=cell_font_size, family="monospace"),
                   height=28))])
    fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=60, b=20)},
                      title=title,
                      height=36 + 28*n_rows + height_extra)
    return fig

# -------------------------------------------------------------------
# Contingency tables (visit & conversion)
# -------------------------------------------------------------------
def build_contingency_card(outcome, label):
    ct = pd.crosstab(df_blog["segment"], df_blog[outcome], margins=True, margins_name="All")
    ct = ct.reindex(SEGMENT_ORDER + ["All"])
    # Build rows
    seg_col = list(ct.index)
    col0 = [f"{v:,}" for v in ct.iloc[:,0].values]
    col1 = [f"{v:,}" for v in ct.iloc[:,1].values]
    col2 = [f"{v:,}" for v in ct.iloc[:,2].values]
    # Proportion rows (excluding margin)
    ct_prop = pd.crosstab(df_blog["segment"], df_blog[outcome], normalize="index")
    ct_prop = ct_prop.reindex(SEGMENT_ORDER)
    prop0 = [f"{v:.4f}" for v in ct_prop.iloc[:,0].values] + ["—"]
    prop1 = [f"{v:.4f}" for v in ct_prop.iloc[:,1].values] + ["—"]

    fig = table_card(
        f"Contingency Table — {label.upper()} × Treatment Group",
        ["Segment", f"{label} = 0 (count)", f"{label} = 1 (count)", "Total",
         f"P({label}=0)", f"P({label}=1)"],
        [seg_col, col0, col1, col2, prop0, prop1],
        [180, 140, 140, 120, 110, 110])
    fig.write_html(os.path.join(OUT_DIR, f"nb03_contingency_{outcome}_interactive.html"), **PLOTLY_KW)
    fig.show()
    return ct

ct_visit = build_contingency_card("visit", "Visit")
ct_conv  = build_contingency_card("conversion", "Conversion")

# -------------------------------------------------------------------
# Chi-square test result cards (visit & conversion)
# -------------------------------------------------------------------
def build_chi2_card(ct, outcome_name):
    # Drop the margin row/col for the test
    core = ct.loc[SEGMENT_ORDER, [c for c in ct.columns if c != "All"]]
    chi2, p, dof, exp = chi2_contingency(core.values)
    sig = p < 0.05
    rows = [
        ("Null hypothesis (H₀)",
         f"Treatment group and {outcome_name} are independent"),
        ("Alt. hypothesis (H₁)",
         f"Treatment group and {outcome_name} are related"),
        ("Chi-square statistic", f"{chi2:.4f}"),
        ("Degrees of freedom", f"{dof}"),
        ("P-value", f"{p:.3e}" if p < 1e-4 else f"{p:.6f}"),
        ("Significance level (α)", "0.05"),
        ("Decision", ("✓ REJECT H₀" if sig else "✗ fail to reject H₀")),
        ("Conclusion", (f"Significant association between treatment and {outcome_name} (p < α)"
                        if sig else f"No significant association (p ≥ α)")),
    ]
    labels = [r[0] for r in rows]; values = [r[1] for r in rows]
    fig = table_card(
        f"Chi-Square Test — Treatment × {outcome_name.capitalize()}",
        ["Field", "Value"], [labels, values], [260, 520])
    fig.write_html(os.path.join(OUT_DIR, f"nb03_chi2_test_{outcome_name}_interactive.html"), **PLOTLY_KW)
    fig.show()
    return chi2, p, dof, exp

chi2_v, p_v, dof_v, exp_v = build_chi2_card(ct_visit, "visit")
chi2_c, p_c, dof_c, exp_c = build_chi2_card(ct_conv,  "conversion")

# -------------------------------------------------------------------
# Expected frequencies table
# -------------------------------------------------------------------
def build_expected_card(exp, outcome_name, cols):
    min_exp = exp.min()
    n_low = int((exp < 5).sum())
    valid = "✓ Yes" if n_low == 0 else "✗ No"
    seg_col = SEGMENT_ORDER
    c0 = [f"{v:,.2f}" for v in exp[:,0]]
    c1 = [f"{v:,.2f}" for v in exp[:,1]]
    fig = table_card(
        f"Expected Frequencies — {outcome_name.capitalize()}",
        ["Segment", f"{outcome_name}=0", f"{outcome_name}=1"],
        [seg_col, c0, c1], [180, 180, 180])
    # Append a summary sub-table
    summary_labels = ["Minimum expected frequency", "Cells with expected < 5", "Valid for chi-square?"]
    summary_values = [f"{min_exp:,.2f}", f"{n_low}", valid]
    # Save as one combined figure using subplots — simpler: add annotations below
    fig.write_html(os.path.join(OUT_DIR, f"nb03_expected_freq_{outcome_name}_interactive.html"), **PLOTLY_KW)
    fig.show()
    # Companion validity card
    fig2 = table_card(
        f"Assumption Check — Expected Frequencies ({outcome_name.capitalize()})",
        ["Check", "Value"], [summary_labels, summary_values], [300, 200])
    fig2.write_html(os.path.join(OUT_DIR, f"nb03_expected_check_{outcome_name}_interactive.html"), **PLOTLY_KW)
    fig2.show()

build_expected_card(exp_v, "visit",      SEGMENT_ORDER)
build_expected_card(exp_c, "conversion", SEGMENT_ORDER)

# -------------------------------------------------------------------
# Pairwise chi-square tests (conversion)
# -------------------------------------------------------------------
def pairwise_chi2(seg1, seg2, outcome):
    sub = df_blog[df_blog["segment"].isin([seg1, seg2])]
    ct = pd.crosstab(sub["segment"], sub[outcome])
    chi2, p, dof, _ = chi2_contingency(ct.values)
    return chi2, p

pairs = [("Mens E-Mail","No E-Mail"), ("Womens E-Mail","No E-Mail"), ("Mens E-Mail","Womens E-Mail")]
labels, chi_vals, p_vals, sig_vals = [], [], [], []
alpha_bonf = 0.05 / 3  # 3 pairwise comparisons
fill_sig = []
for a,b in pairs:
    chi_v, p_v_ = pairwise_chi2(a, b, "conversion")
    labels.append(f"{a} vs {b}")
    chi_vals.append(f"{chi_v:.4f}")
    p_vals.append(f"{p_v_:.3e}" if p_v_ < 1e-4 else f"{p_v_:.6f}")
    is_sig = p_v_ < alpha_bonf
    sig_vals.append("Yes" if is_sig else "No")
    fill_sig.append("#D5F5E3" if is_sig else "#FADBD8")

stripe = ["#F8F9F9" if i%2==0 else "white" for i in range(3)]
fig = go.Figure(data=[go.Table(
    columnwidth=[300, 120, 140, 140, 130],
    header=dict(values=[f"<b>{h}</b>" for h in ["Comparison","χ²","P-value","Bonferroni α","Significant?"]],
                fill_color="#2C3E50", font=dict(color="white", size=13), align="center", height=36),
    cells=dict(values=[labels, chi_vals, p_vals, [f"{alpha_bonf:.4f}"]*3, sig_vals],
               fill_color=[stripe, stripe, stripe, stripe, fill_sig],
               align="center", font=dict(size=12, family="monospace"), height=30))])
fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=60, b=20)},
                  title="Pairwise Chi-Square Tests — Conversion",
                  height=36 + 30*3 + 80)
fig.write_html(os.path.join(OUT_DIR, "nb03_pairwise_chi2_interactive.html"), **PLOTLY_KW)
fig.show()

# -------------------------------------------------------------------
# Cramér's V effect size
# -------------------------------------------------------------------
def cramers_v(chi2, n, min_dim):
    return np.sqrt(chi2 / (n * (min_dim - 1)))

def v_interp(v):
    if v < 0.1: return "Very weak"
    if v < 0.2: return "Weak"
    if v < 0.4: return "Moderate"
    if v < 0.6: return "Strong"
    return "Very strong"

n = len(df_blog)
min_dim = min(3, 2)  # 3 segments × 2 outcomes → min(3,2)-1=1
v_visit = cramers_v(chi2_v, n, min_dim)
v_conv  = cramers_v(chi2_c, n, min_dim)

fig = table_card(
    "Effect Size — Cramér's V",
    ["Outcome", "Cramér's V", "Interpretation"],
    [["Visit","Conversion"],
     [f"{v_visit:.4f}", f"{v_conv:.4f}"],
     [v_interp(v_visit), v_interp(v_conv)]],
    [200, 180, 260])
fig.write_html(os.path.join(OUT_DIR, "nb03_cramers_v_interactive.html"), **PLOTLY_KW)
fig.show()

# -------------------------------------------------------------------
# Standardized residuals (visit & conversion)
# -------------------------------------------------------------------
def build_residuals_card(ct, exp, outcome_name):
    core = ct.loc[SEGMENT_ORDER, [c for c in ct.columns if c != "All"]]
    resid = (core.values - exp) / np.sqrt(exp)
    # color cells by |resid| > 2
    c0 = [f"{v:+.3f}" for v in resid[:,0]]
    c1 = [f"{v:+.3f}" for v in resid[:,1]]
    def colorize(vals):
        return ["#FADBD8" if abs(v) > 2 else "#F8F9F9" for v in vals]
    stripe_seg = ["#F8F9F9"]*len(SEGMENT_ORDER)
    fill_c0 = colorize(resid[:,0])
    fill_c1 = colorize(resid[:,1])
    fig = go.Figure(data=[go.Table(
        columnwidth=[220, 180, 180],
        header=dict(values=[f"<b>{h}</b>" for h in ["Segment", f"{outcome_name}=0", f"{outcome_name}=1"]],
                    fill_color="#2C3E50", font=dict(color="white", size=13), align="center", height=36),
        cells=dict(values=[SEGMENT_ORDER, c0, c1],
                   fill_color=[stripe_seg, fill_c0, fill_c1],
                   align="center", font=dict(size=12, family="monospace"), height=30))])
    fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=70, b=20)},
                      title=f"Standardized Residuals — {outcome_name.capitalize()} (red cells: |residual| > 2)",
                      height=36 + 30*len(SEGMENT_ORDER) + 100)
    fig.write_html(os.path.join(OUT_DIR, f"nb03_residuals_{outcome_name}_interactive.html"), **PLOTLY_KW)
    fig.show()

build_residuals_card(ct_visit, exp_v, "visit")
build_residuals_card(ct_conv,  exp_c, "conversion")

# -------------------------------------------------------------------
# Overall chi-square summary
# -------------------------------------------------------------------
rows = [
    ("Overall: Treatment × Visit",      chi2_v, dof_v, p_v, v_visit, v_interp(v_visit)),
    ("Overall: Treatment × Conversion", chi2_c, dof_c, p_c, v_conv,  v_interp(v_conv)),
]
sig_col = ["Yes" if r[3]<0.05 else "No" for r in rows]
sig_fill = ["#D5F5E3" if s=="Yes" else "#FADBD8" for s in sig_col]
stripe = ["#F8F9F9" if i%2==0 else "white" for i in range(len(rows))]
fig = go.Figure(data=[go.Table(
    columnwidth=[260, 120, 90, 140, 140, 160, 140],
    header=dict(values=[f"<b>{h}</b>" for h in
                        ["Test","Chi-Square","DF","P-value","Cramér's V","Effect","Significant?"]],
                fill_color="#2C3E50", font=dict(color="white", size=13), align="center", height=36),
    cells=dict(values=[
        [r[0] for r in rows],
        [f"{r[1]:.4f}" for r in rows],
        [str(r[2]) for r in rows],
        [f"{r[3]:.3e}" if r[3]<1e-4 else f"{r[3]:.6f}" for r in rows],
        [f"{r[4]:.4f}" for r in rows],
        [r[5] for r in rows],
        sig_col,
    ],
    fill_color=[stripe]*6 + [sig_fill],
    align="center", font=dict(size=12, family="monospace"), height=30))])
fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=60, b=20)},
                  title="Chi-Square Test Summary — All Outcomes",
                  height=36 + 30*len(rows) + 100)
fig.write_html(os.path.join(OUT_DIR, "nb03_chi2_summary_interactive.html"), **PLOTLY_KW)
fig.show()
print("\n  ✓ All nb03 results-display cards saved to:", OUT_DIR)



  ✓ All nb03 results-display cards saved to: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/ab_testing/data/outputs/nb03
